# Part 1 — Python and data foundations
Run from the repository root. The original files in `data/` are never edited.

In [ ]:
import csv, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
ROOT = Path.cwd()
DATA = ROOT / 'data'
assert (DATA / 'appointments.csv').exists(), 'Run from repository root'


## 1.1 CSV module only — independent of pandas cleaning

In [ ]:
accepted, rejected = [], []
def parse_row(row):
    converted = row.copy()
    for field in ('age', 'fee_inr', 'prior_no_shows'):
        converted[field] = int(row[field])
    converted['distance_km'] = float(row['distance_km'])
    return converted
with open(DATA / 'appointments.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    print('Header:', reader.fieldnames)
    rows = list(reader)
print('Raw row count:', len(rows))
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)
with open(ROOT / 'rejected_rows.json', 'w', encoding='utf-8') as f:
    json.dump(rejected, f, indent=2)
print('Accepted:', len(accepted), 'Rejected:', len(rejected))
assert (len(rows), len(accepted), len(rejected)) == (1020, 988, 32)


## 1.2 Original CSV — cleaning pass 1

In [ ]:
df = pd.read_csv(DATA / 'appointments.csv')
print('Original shape:', df.shape)
df.info()
print('Nulls:', df.isnull().sum().to_dict())
print('Exact duplicates:', df.duplicated().sum())
print('Raw booking channel spellings:', df['booking_channel'].nunique())
df = df.drop_duplicates().copy()
print('After dropping exact duplicates:', len(df))
df['booking_channel'] = df['booking_channel'].str.strip().str.title()
print('Clean booking channels:', sorted(df['booking_channel'].unique()))
df['appt_date'] = pd.to_datetime(df['appt_date'], errors='coerce')
df['booked_on'] = pd.to_datetime(df['booked_on'])
df = df.dropna(subset=['appt_date', 'status']).copy()
print('After dropping invalid date / missing status:', len(df))
assert len(df) == 990


## 1.3 Validity, missing data and IQR clipping

In [ ]:
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan
print('Missing age after validity:', df['age'].isna().sum())
median_age = df['age'].median()
median_distance = df['distance_km'].median()
print('Median age:', median_age, 'median distance:', median_distance)
df['age'] = df['age'].fillna(median_age).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(median_distance)
for col in ['distance_km', 'fee_inr']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_capped = int(((df[col] < low) | (df[col] > high)).sum())
    print(f'{col}: Q1={q1}, Q3={q3}, lower={low}, upper={high}, capped={n_capped}')
    df[col] = np.clip(df[col], low, high)
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)


## 1.4 Features and vectorised NumPy summary

In [ ]:
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = (df['status'] == 'No-Show').astype(int)
X = df[['age', 'distance_km', 'fee_inr', 'lead_days']].to_numpy()
print('Means [age, distance, fee, lead_days]:', X.mean(axis=0))
mask = X[:, 3] > 14
print('Lead >14 days:', int(mask.sum()), 'share:', float(mask.mean()))
z = (X[:, 1] - np.mean(X[:, 1])) / np.std(X[:, 1])
print('Distance |z| > 2:', int((np.abs(z) > 2).sum()))
print('No-shows:', int(df['no_show'].sum()), 'rate:', round(df['no_show'].mean()*100, 1))


## 1.5 Open-Meteo API: retry once and never fabricate readings

In [ ]:
clinics = pd.read_csv(DATA / 'clinics.csv')
def fetch_temperature(clinic):
    params = {'latitude': float(clinic.latitude), 'longitude': float(clinic.longitude),
              'hourly': 'temperature_2m', 'forecast_days': 1}
    for attempt in range(2):
        try:
            response = requests.get('https://api.open-meteo.com/v1/forecast',
                                    params=params, timeout=10)
            if response.status_code != 200:
                raise requests.exceptions.HTTPError(f'HTTP {response.status_code}')
            values = response.json().get('hourly', {}).get('temperature_2m')
            if not values or values[0] is None:
                raise ValueError('hourly.temperature_2m missing')
            return float(values[0])
        except (requests.exceptions.RequestException, ValueError, KeyError, TypeError) as exc:
            print(f'{clinic.clinic_name}, attempt {attempt+1}: {exc}')
    return None
clinics['temp_c'] = [fetch_temperature(row) for row in clinics.itertuples(index=False)]
clinics.to_csv(ROOT / 'clinics_enriched.csv', index=False)
print(clinics[['clinic_name', 'temp_c']].to_string(index=False))


## 1.6 Export 16 columns, validate, and generate SQL INSERTs

In [ ]:
columns = ['appointment_id','patient_id','clinic_id','booked_on','appt_date','age','gender',
           'distance_km','fee_inr','booking_channel','visit_type','reminder_sent',
           'prior_no_shows','status','lead_days','no_show']
df = df.sort_values('appointment_id').copy()
assert df['appointment_id'].is_unique
assert df[columns].isna().sum().sum() == 0
for col in ('booked_on', 'appt_date'):
    df[col] = df[col].dt.strftime('%Y-%m-%d')
df[columns].to_csv(ROOT / 'appointments_clean.csv', index=False)
def sql_value(v):
    if pd.isna(v):
        return 'NULL'
    if isinstance(v, (str, np.str_)):
        return "'" + str(v).replace("'", "''") + "'"
    if isinstance(v, (int, float, np.integer, np.floating)):
        return str(v)
    raise TypeError(f'Unexpected value type: {type(v)}')
with open(ROOT / 'load_data.sql', 'w', encoding='utf-8') as f:
    f.write('-- Run 02_queries.sql table creation first; clinics before appointments.\n')
    for table, frame, cols in [('clinics', clinics, list(clinics.columns)),
                               ('appointments', df, columns)]:
        for record in frame[cols].itertuples(index=False, name=None):
            f.write(f"INSERT INTO {table} VALUES (" + ', '.join(sql_value(v) for v in record) + ');\n')
print('Exported:', len(df), 'appointments;', len(clinics), 'clinics')
print('INSERT statements:', sum(line.startswith('INSERT') for line in open(ROOT/'load_data.sql')))
assert len(df) == 990 and df.shape[1] == 16
